In [54]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import lightgbm as lgb
from pathlib import Path


In [55]:
INPUT_PATH = "../data/processed/train_fe.csv"
OUTPUT_DIR = Path("../data/processed/ml_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INPUT_PATH, parse_dates=["date"])
df = df.sort_values(["item_id", "date"]).reset_index(drop=True)


In [56]:
TARGET = "sales"

FEATURES = ["lag_7", "lag_14", "lag_28",
    "rmean_7", "rmean_14", "rmean_28",
    "wday", "month", "year",
    "is_event"]

In [57]:
HORIZON = 28

train_df = df.groupby("item_id").head(-HORIZON)
valid_df = df.groupby("item_id").tail(HORIZON)

x_train = train_df[FEATURES]
y_train = train_df[TARGET]

x_valid = valid_df[FEATURES]
y_valid = valid_df[TARGET]

x_tune = x_train.iloc[:100_000]
y_tune = y_train.iloc[:100_000]


In [58]:
def evaluate(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred))
    }


## Linear Regression

In [59]:
lr = LinearRegression()
lr.fit(x_train, y_train)
valid_df["pred_lr"] = lr.predict(x_valid)
lr_metrics = evaluate(y_valid, valid_df["pred_lr"])
print("LR:", lr_metrics)

LR: {'MAE': 1.1059775935482958, 'RMSE': np.float64(2.493874513635676)}


## Ridge Regression

In [60]:
ridge = Ridge()
ridge.fit(x_train, y_train)
valid_df["pred_ridge"] = ridge.predict(x_valid)
ridge_metrics = evaluate(y_valid, valid_df["pred_ridge"])
print("Ridge:", ridge_metrics)

Ridge: {'MAE': 1.1059775992547385, 'RMSE': np.float64(2.4938745379293676)}


## Random Forest Regressor

In [61]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=10,
    n_jobs=1,
    random_state=42
)

rf.fit(x_tune, y_tune)
valid_df["pred_rf"] = rf.predict(x_valid)
rf_metrics = evaluate(y_valid, valid_df["pred_rf"])
print("RF:", rf_metrics)

RF: {'MAE': 1.1552526004622474, 'RMSE': np.float64(2.7686216406001414)}


## XGBoost Regressor

In [62]:
xgb_model = XGBRegressor(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    tree_method="hist",
    eval_metric="mae",
    random_state=42,
    n_jobs=4
)

xgb_model.fit(
    x_train,
    y_train,
    eval_set=[(x_valid, y_valid)]
)

valid_df["pred_xgb"] = xgb_model.predict(x_valid)
xgb_metrics = evaluate(y_valid, valid_df["pred_xgb"])
print("XGB:", xgb_metrics)

[0]	validation_0-mae:1.45186
[1]	validation_0-mae:1.42069
[2]	validation_0-mae:1.39472
[3]	validation_0-mae:1.37160
[4]	validation_0-mae:1.35103
[5]	validation_0-mae:1.33179
[6]	validation_0-mae:1.31467
[7]	validation_0-mae:1.29863
[8]	validation_0-mae:1.28386
[9]	validation_0-mae:1.27067
[10]	validation_0-mae:1.25857
[11]	validation_0-mae:1.24758
[12]	validation_0-mae:1.23769
[13]	validation_0-mae:1.22842
[14]	validation_0-mae:1.22005
[15]	validation_0-mae:1.21230
[16]	validation_0-mae:1.20527
[17]	validation_0-mae:1.19877
[18]	validation_0-mae:1.19273
[19]	validation_0-mae:1.18733
[20]	validation_0-mae:1.18229
[21]	validation_0-mae:1.17768
[22]	validation_0-mae:1.17330
[23]	validation_0-mae:1.16912
[24]	validation_0-mae:1.16540
[25]	validation_0-mae:1.16187
[26]	validation_0-mae:1.15853
[27]	validation_0-mae:1.15556
[28]	validation_0-mae:1.15282
[29]	validation_0-mae:1.15024
[30]	validation_0-mae:1.14790
[31]	validation_0-mae:1.14586
[32]	validation_0-mae:1.14391
[33]	validation_0-ma

## Hyperparameter Tuning for XGBoost Regressor

In [63]:
xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    tree_method='hist',
    eval_metric="mae",
    n_jobs=1
)

xgb_param_dist = {
    "n_estimators": [300,400,500,600],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 5, 7],
    "min_child_weight": [1, 5, 10],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "gamma": [0, 0.1, 0.3],
    "reg_alpha": [0, 0.1, 1],
    "reg_lambda": [1, 5, 10],
}

xgb_search = RandomizedSearchCV(
    xgb,
    param_distributions=xgb_param_dist,
    n_iter=10,
    cv=TimeSeriesSplit(n_splits=2),
    scoring="neg_mean_absolute_error",
    n_jobs=1,
    random_state=42
)

xgb_search.fit(x_tune,y_tune)
best_xgb = xgb_search.best_estimator_

best_xgb.fit(
    x_train,
    y_train,
    eval_set=[(x_valid, y_valid)]
)

valid_df["pred_best_xgb"] = best_xgb.predict(x_valid)
best_xgb_metrics = evaluate(y_valid, valid_df["pred_best_xgb"])
print("Best XGB:", best_xgb_metrics)

[0]	validation_0-mae:1.47963
[1]	validation_0-mae:1.47360
[2]	validation_0-mae:1.46678
[3]	validation_0-mae:1.46014
[4]	validation_0-mae:1.45374
[5]	validation_0-mae:1.44716
[6]	validation_0-mae:1.44086
[7]	validation_0-mae:1.43464
[8]	validation_0-mae:1.42875
[9]	validation_0-mae:1.42300
[10]	validation_0-mae:1.41749
[11]	validation_0-mae:1.41223
[12]	validation_0-mae:1.40877
[13]	validation_0-mae:1.40359
[14]	validation_0-mae:1.39860
[15]	validation_0-mae:1.39368
[16]	validation_0-mae:1.38908
[17]	validation_0-mae:1.38504
[18]	validation_0-mae:1.38047
[19]	validation_0-mae:1.37613
[20]	validation_0-mae:1.37180
[21]	validation_0-mae:1.36868
[22]	validation_0-mae:1.36500
[23]	validation_0-mae:1.36078
[24]	validation_0-mae:1.35666
[25]	validation_0-mae:1.35275
[26]	validation_0-mae:1.34888
[27]	validation_0-mae:1.34513
[28]	validation_0-mae:1.34131
[29]	validation_0-mae:1.33806
[30]	validation_0-mae:1.33442
[31]	validation_0-mae:1.33076
[32]	validation_0-mae:1.32724
[33]	validation_0-ma

## LightGBM Regressor

In [64]:
lgbm_model = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="l1",
    random_state=42,
    n_jobs=4
)

lgbm_model.fit(
    x_train,
    y_train,
    eval_set=[(x_valid, y_valid)]
)

valid_df["pred_lgbm"] = lgbm_model.predict(x_valid)
lgb_metrics =  evaluate(y_valid, valid_df["pred_lgbm"])
print("LGBM:", lgbm_metrics)

[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.112564 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1066
[LightGBM] [Info] Number of data points in the train set: 4610088, number of used features: 10
[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Info] Start training from score 1.066998
[LightGBM] [Warning] Unknown parameter: eval_metric
LGBM: {'MAE': 1.1169740092380394, 'RMSE': np.float64(2.488074738117806)}


## Hyperparameter Tuning for LightGBM Regressor

In [65]:
lgbm = LGBMRegressor(
    objective="regression",
    eval_metric="l1",
    random_state=42,
    n_jobs=1
)

lgbm_param_dist = {
    "n_estimators": [300,400,500,600],
    "learning_rate": [0.01, 0.05, 0.1],
    "num_leaves": [31, 63, 127],
    "max_depth": [5, 8, 12, -1],
    "min_child_samples": [20, 50, 100],
    "feature_fraction": [0.6, 0.8, 1.0],
    "bagging_fraction": [0.6, 0.8, 1.0],
    "bagging_freq": [1],
    "lambda_l1": [0, 0.1, 1],
    "lambda_l2": [0, 1, 5]
}

lgbm_search = RandomizedSearchCV(
    lgbm,
    param_distributions=lgbm_param_dist,
    n_iter=6,
    cv=TimeSeriesSplit(n_splits=2),
    scoring="neg_mean_absolute_error",
    n_jobs=1,
    random_state=42
)

lgbm_search.fit(x_tune, y_tune)
best_lgbm = lgbm_search.best_estimator_

best_lgbm.fit(
    x_train,
    y_train,
    eval_set=[(x_valid, y_valid)]
)

valid_df["pred_best_lgbm"] = best_lgbm.predict(x_valid)
best_lgbm_metrics = evaluate(y_valid, valid_df["pred_best_lgbm"])
print("Best LGBM:", best_lgbm_metrics)

[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] lambda_l1 is set=0, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0
[LightGBM] [Warning] lambda_l2 is set=0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] lambda_l1 is set=0, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0
[LightGBM] [Warning] lambda_l2 is set=0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0
[LightGBM] [Warning] bagg

In [66]:
all_metrics = [
    {"model": "LinearRegression", **lr_metrics},
    {"model": "Ridge", **ridge_metrics},
    {"model": "RandomForest", **rf_metrics},
    {"model": "BestRandomForest", **best_rf_metrics},
    {"model": "XGBoost", **xgb_metrics},
    {"model": "BestXGBoost", **best_xgb_metrics},
    {"model": "LightGBM", **lgbm_metrics},
    {"model": "BestLightGBM", **best_lgbm_metrics},
]


In [67]:
metrics_df = (
    pd.DataFrame(all_metrics)
      .sort_values(by=["MAE", "RMSE"], ascending=[True, True])
      .reset_index(drop=True)
)

metrics_df


,model,MAE,RMSE
0,LinearRegression,1.105978,2.493875
1,Ridge,1.105978,2.493875
2,BestLightGBM,1.109974,2.473644
3,BestXGBoost,1.116479,2.532847
4,BestRandomForest,1.116625,2.475755
5,LightGBM,1.116974,2.488075
6,XGBoost,1.121194,2.533151
7,RandomForest,1.155253,2.768622


In [68]:
metrics_df.to_csv(
    OUTPUT_DIR / "ml_model_comparison_metrics.csv",
    index=False
)


In [69]:
pred_cols = [
    "date", "store_id", "item_id", "sales",
    "pred_lr",
    "pred_best_xgb",
    "pred_best_lgbm"
]

valid_df[pred_cols].to_csv(
    OUTPUT_DIR / "ml_model_predictions.csv",
    index=False
)
